# Qmaze: a rat learns a maze (Q-table, then a Keras network), every line tied to its idea

Same style as the CartPole notebook: small pieces first, every piece run, and the connections are written **inside the code comments** so you don't have to scroll.

| Tag in a comment | Meaning |
|---|---|
| `# WHAT:` | the line below, in plain words |
| `# WHY:` | why it exists, its significance |
| `# SAME AS:` | the same idea somewhere else (the other line is quoted right there) |
| `# BECOMES:` | what it turns into in the next, fancier version |
| `# WATCH:` | a trap |

**The version of the problem used here.** This is the well-known version of Qmaze: a rat starts in the top-left of a grid, walls are `0` and open cells are `1`, there are four moves (left, up, right, down), and the cheese is in the bottom-right corner. Other versions change the details, not the ideas.

### The same five jobs as before
| # | Job | In the maze |
|---|---|---|
| 1 | **Environment loop** | the rat moves, the maze answers with a reward and a new position |
| 2 | **Policy** | the rule that turns "where am I" into "which way do I go" |
| 3 | **Target** | the answer key: what the number *should* have been |
| 4 | **Update** | move the learned numbers toward the target |
| 5 | **Exploration** | sometimes try a random move |

Results use random numbers, so your numbers will differ a little run to run. Every number in the text below the cells is printed by the code, not typed by me.

# Part 1: The maze is a game, and a random rat is the "before" score

> **You are here:** nothing is learned yet.
> **State** = which cell the rat is in, written `(row, col)`. **Action** = left, up, right or down. **Reward** = points for the move just made.
> **Why a random rat is in here:** an agent that has learned nothing behaves like a coin flip. Its score is the **"before"** number. Everything later tries to move the score up from there, and without it you couldn't tell if learning did anything.

Rewards used (the usual Qmaze values): **-0.04** for an ordinary move (a small cost for wandering), **-0.75** for bumping into a wall or the edge (you stay put), **+1.0** for reaching the cheese. The game is **lost** if the running total drops below `-0.5 * (number of cells)`, which stops rats that wander forever.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"        # WHAT: hide TensorFlow's startup noise (it only gets loaded in Part 5)
import random, time, collections
import numpy as np

LEFT, UP, RIGHT, DOWN = 0, 1, 2, 3                # WHAT: the four actions are just the numbers 0 to 3
MOVES  = {LEFT: (0, -1), UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0)}   # WHAT: (row change, column change) for each action
ARROWS = {LEFT: "←", UP: "↑", RIGHT: "→", DOWN: "↓"}
NAMES  = {LEFT: "left", UP: "up", RIGHT: "right", DOWN: "down"}


class Maze:
    # WHAT: the game. grid[r][c] is 1 for an open cell and 0 for a wall. The cheese is the bottom-right cell.
    # WHY:  this class is JOB 1 (the environment). The agent never looks inside it; it only calls reset() and step().
    # SAME AS: CartPole's `env.reset()` / `env.step(action)`. (The popular Qmaze tutorial calls its version `Qmaze`.)

    def __init__(self, grid, slip=0.0):
        self.grid = np.array(grid, dtype=float)
        self.nrows, self.ncols = self.grid.shape
        self.target = (self.nrows - 1, self.ncols - 1)
        self.min_reward = -0.5 * self.grid.size       # WHAT: the "give up" line: lose if the total reward falls below this
        self.slip = slip                              # WHAT: chance a move goes in a random direction (0 = predictable maze). Used in Part 4.
        self.reset()

    def reset(self, start=(0, 0)):
        self.pos = start                              # WHAT: where the rat is. THIS is the state.
        self.total = 0.0
        self.status = "playing"
        return self.pos                               # SAME AS: CartPole `obs, info = env.reset()`

    def step(self, action):
        if random.random() < self.slip:
            action = random.randrange(4)              # WHAT: a "slippery floor": sometimes you go the wrong way
        r, c = self.pos
        dr, dc = MOVES[action]
        nr, nc = r + dr, c + dc
        inside = (0 <= nr < self.nrows) and (0 <= nc < self.ncols)
        if (not inside) or self.grid[nr, nc] == 0:
            reward = -0.75                            # WHAT: bumped into a wall or the edge. The rat stays where it is.
        else:
            self.pos = (nr, nc)                       # WHAT: a legal move: the rat actually moves
            reward = 1.0 if self.pos == self.target else -0.04
        self.total = self.total + reward
        if self.pos == self.target:
            self.status = "win"
        elif self.total < self.min_reward:
            self.status = "lose"                      # WATCH: this is really "ran out of patience". Strictly it's a time-out, not a true
                                                      #        ending, but the popular Qmaze code (and this notebook) treats it as game over.
        else:
            self.status = "playing"
        return reward, self.status                    # SAME AS: CartPole `reward, terminated/truncated`


GRID3 = [[1, 1, 0],
         [0, 1, 1],
         [0, 0, 1]]                                   # WHAT: a tiny 3x3 maze, small enough to check every number by hand

GRID7 = [[1, 1, 0, 1, 1, 1, 1],
         [0, 1, 0, 1, 0, 0, 1],
         [1, 1, 1, 1, 0, 1, 1],
         [1, 0, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 0, 1, 1],
         [0, 0, 0, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1, 1]]                       # WHAT: the "real" 7x7 maze. It has a short route and a long detour on purpose.


def show(grid, path=(), rat=None):
    # WHAT: draw the maze as text. █ = wall, · = open, C = cheese, R = rat, * = a cell on a path.
    n, m = np.array(grid).shape
    for r in range(n):
        row = ""
        for c in range(m):
            if (r, c) == (n - 1, m - 1):  row += " C"
            elif (r, c) == rat:           row += " R"
            elif (r, c) in path:          row += " *"
            elif grid[r][c] == 0:         row += " █"
            else:                         row += " ·"
        print(row)


def shortest_path_length(grid, start=(0, 0)):
    # WHAT: breadth-first search = the true shortest route. We use it ONLY as a reference to grade the learner;
    #       the learner is never told this.
    g = np.array(grid); n, m = g.shape; goal = (n - 1, m - 1)
    queue = collections.deque([(start, 0)]); seen = {start}
    while queue:
        (r, c), d = queue.popleft()
        if (r, c) == goal:
            return d
        for dr, dc in MOVES.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < m and g[nr, nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc)); queue.append(((nr, nc), d + 1))


print("the 3x3 maze:"); show(GRID3, rat=(0, 0))
print()
print("the 7x7 maze:"); show(GRID7, rat=(0, 0))
print()
print("shortest possible route, 3x3:", shortest_path_length(GRID3), "moves")
print("shortest possible route, 7x7:", shortest_path_length(GRID7), "moves")

the 3x3 maze:
 R · █
 █ · ·
 █ █ C

the 7x7 maze:
 R · █ · · · ·
 █ · █ · █ █ ·
 · · · · █ · ·
 · █ █ █ █ · █
 · · · · █ · ·
 █ █ █ · · · █
 · · · █ █ · C

shortest possible route, 3x3: 4 moves
shortest possible route, 7x7: 14 moves


Now the loop. The player below is a coin flip (a random rat). Read each printed line as a sentence: *"the rat was at X, tried a direction, the maze gave this reward, the rat is now at Y."*

In [2]:
random.seed(2)
game = Maze(GRID3)
state = game.reset()                                  # WHAT: the rat starts at (0, 0)
print("start:", state)

while game.status == "playing":
    action = random.randrange(4)                      # WHAT: THE PLAYER: a coin flip over 4 directions.
                                                      # WHY:  THIS ONE LINE is the only thing that differs between a random rat,
                                                      #       a Q-table rat and a Keras rat. Every algorithm below just fills in this line.
    old = game.pos
    reward, status = game.step(action)
    print(f"  at {old}, tried {NAMES[action]:5s} -> now at {game.pos}, reward {reward:+.2f}, total {game.total:+.2f}, status {status}")
    if game.total < -8 and game.status == "playing":
        print("  ... (stopping the printout early; a random rat can wander for a long time)")
        break

start: (0, 0)
  at (0, 0), tried left  -> now at (0, 0), reward -0.75, total -0.75, status playing
  at (0, 0), tried right -> now at (0, 1), reward -0.04, total -0.79, status playing
  at (0, 1), tried right -> now at (0, 1), reward -0.75, total -1.54, status playing
  at (0, 1), tried up    -> now at (0, 1), reward -0.75, total -2.29, status playing
  at (0, 1), tried up    -> now at (0, 1), reward -0.75, total -3.04, status playing
  at (0, 1), tried down  -> now at (1, 1), reward -0.04, total -3.08, status playing
  at (1, 1), tried right -> now at (1, 2), reward -0.04, total -3.12, status playing
  at (1, 2), tried down  -> now at (2, 2), reward +1.00, total -2.12, status win


**The ruler.** To compare players we play many games and count two things: the **win rate** (fraction of games that reach the cheese) and the **average number of moves**. `play_many` takes any function that turns a state into an action, so the same ruler works for the random rat, the table rat and the Keras rat.

In [3]:
def play_many(policy, grid, games=300, slip=0.0, cap=200):
    # WHAT: play `games` complete games with `policy` (a function: state -> action). Return (win rate, average moves).
    # WHY:  the ONE ruler for every player in this notebook, so the numbers are comparable.
    # SAME AS: CartPole's `evaluate_cartpole(policy)`.
    wins, lengths = 0, []
    for _ in range(games):
        env = Maze(grid, slip=slip)
        state = env.reset()
        n = 0
        while env.status == "playing" and n < cap:
            env.step(policy(state))                   # SAME AS: `action = policy(state)`: the plug-in point
            state = env.pos
            n += 1
        wins += (env.status == "win")
        lengths.append(n)
    return wins / games, float(np.mean(lengths))


def random_policy(state):
    # WHAT: ignores the state and picks any direction.
    # WHY:  THIS IS THE "BEFORE" SCORE (an agent that knows nothing). BECOMES: the Q-table starts all zeros, so its first choices
    #       are random too (Part 2), and epsilon = 1.0 / an untrained network is random as well (Part 5).
    return random.randrange(4)


random.seed(0)
before_3 = play_many(random_policy, GRID3)
before_7 = play_many(random_policy, GRID7)
print(f"random rat, 3x3: wins {before_3[0]*100:5.1f}% of games, average {before_3[1]:5.1f} moves per game")
print(f"random rat, 7x7: wins {before_7[0]*100:5.1f}% of games, average {before_7[1]:5.1f} moves per game")
print(f"for reference the shortest routes are {shortest_path_length(GRID3)} and {shortest_path_length(GRID7)} moves")

random rat, 3x3: wins   7.7% of games, average   9.1 moves per game
random rat, 7x7: wins   1.0% of games, average  58.0 moves per game
for reference the shortest routes are 4 and 14 moves


**What the output means.** The random rat rarely finishes the big maze, and when it fails it burns a lot of moves. The gap between this score and "win every game in the shortest number of moves" is the whole job of learning.

# Part 2: The Q-table by hand on the 3x3 maze

> **You are here:** you have a ruler and a "before" score. Now the first real learner.
> **New idea:** a table with **one row per cell** the rat can stand in and **one column per direction**. Each number answers: *"if the rat is in this cell and goes this way, about how many points will it collect from here to the end?"*

This answers "what are S0, S1?" from earlier: in a maze the **states are just the squares on the picture**, so I label them by their `(row, col)`. The 3x3 maze has 9 squares and 4 directions, so the table has 9 × 4 = 36 numbers (walls are never visited). It starts at **all zeros**: the rat knows nothing.

**The update, after every move (it changes exactly one number: the (square, direction) just used):**
1. `target = reward + gamma * (the biggest of the 4 numbers in the NEW square's row)` is the **answer key**: *one real reward just received, plus what the next row says is still to come.*
2. `new = old + alpha * (target - old)` moves that one number **part of the way** to the answer key. `alpha` is the fraction (0.5 = halfway) and `gamma` is how much the future counts (0.9 = 90%).

First the arithmetic once, with pretend numbers, and then on the real maze.

In [4]:
alpha, gamma = 0.5, 0.9

# PRETEND numbers, just to watch the arithmetic one time:
old_value   = 0.0      # the table number for (square (1,1), go RIGHT), currently still 0
reward      = -0.04    # an ordinary move
best_next   = 0.80     # the best of the 4 numbers in the row for the NEW square (1,2)

target = reward + gamma * best_next
gap    = target - old_value
new    = old_value + alpha * gap
print(f"target = reward + gamma * best_next = {reward} + {gamma} * {best_next} = {target:.2f}")
print(f"gap    = target - old               = {target:.2f} - {old_value} = {gap:.2f}")
print(f"new    = old + alpha * gap          = {old_value} + {alpha} * {gap:.2f} = {new:.2f}")

target = reward + gamma * best_next = -0.04 + 0.9 * 0.8 = 0.68
gap    = target - old               = 0.68 - 0.0 = 0.68
new    = old + alpha * gap          = 0.0 + 0.5 * 0.68 = 0.34


**Now on the real 3x3 maze.** To see information travel, the rat below is told to walk the solution route right, down, right, down, three laps in a row. (A real learner discovers a route by itself; Part 3 does that. Here I force it so every number is easy to follow.) Only the numbers on the route are printed.

In [5]:
alpha, gamma = 0.5, 0.9

# WHAT: the Q-table as a dictionary: key = (row, col, action), value = the score. Everything starts at 0.0.
# WHY:  THIS DICTIONARY *IS* THE AGENT'S KNOWLEDGE. Learning = changing the numbers in here.
# BECOMES: Part 3 uses a numpy array of shape (rows, cols, 4); Part 5 replaces it with a Keras network's weights.
Q3 = {(r, c, a): 0.0 for r in range(3) for c in range(3) for a in range(4)}

route = [RIGHT, DOWN, RIGHT, DOWN]                     # (0,0) -> (0,1) -> (1,1) -> (1,2) -> (2,2)=cheese

env = Maze(GRID3)
for lap in (1, 2, 3):
    print(f"=== LAP {lap} ===")
    state = env.reset()
    for action in route:
        reward, status = env.step(action)
        next_state = env.pos

        # ---- JOB 3: THE TARGET (the answer key) ----
        if status != "playing":
            best_next = 0.0                              # WHAT: reached the cheese: game over, nothing comes after
        else:
            best_next = max(Q3[next_state + (a,)] for a in range(4))   # WHAT: the best number in the NEW square's row
        target = reward + gamma * best_next              # WHY: "better than the old number" = it contains ONE REAL FACT (the reward)
                                                         #      the old number didn't have. That is the only source of learning.

        # ---- JOB 4: THE UPDATE ----
        key = state + (action,)                          # WHAT: which ONE number we are about to change
        old = Q3[key]
        Q3[key] = old + alpha * (target - old)           # WHAT: move it a fraction alpha toward the target. Nothing else changes.

        print(f"  {state} go {NAMES[action]:5s} -> {next_state} | reward {reward:+.2f} | target = {reward:+.2f} + {gamma}*{best_next:.3f} = {target:+.3f}"
              f" | table[{state}, {NAMES[action]}]: {old:+.3f} -> {Q3[key]:+.3f}")
        state = next_state

=== LAP 1 ===
  (0, 0) go right -> (0, 1) | reward -0.04 | target = -0.04 + 0.9*0.000 = -0.040 | table[(0, 0), right]: +0.000 -> -0.020
  (0, 1) go down  -> (1, 1) | reward -0.04 | target = -0.04 + 0.9*0.000 = -0.040 | table[(0, 1), down]: +0.000 -> -0.020
  (1, 1) go right -> (1, 2) | reward -0.04 | target = -0.04 + 0.9*0.000 = -0.040 | table[(1, 1), right]: +0.000 -> -0.020
  (1, 2) go down  -> (2, 2) | reward +1.00 | target = +1.00 + 0.9*0.000 = +1.000 | table[(1, 2), down]: +0.000 -> +0.500
=== LAP 2 ===
  (0, 0) go right -> (0, 1) | reward -0.04 | target = -0.04 + 0.9*0.000 = -0.040 | table[(0, 0), right]: -0.020 -> -0.030
  (0, 1) go down  -> (1, 1) | reward -0.04 | target = -0.04 + 0.9*0.000 = -0.040 | table[(0, 1), down]: -0.020 -> -0.030
  (1, 1) go right -> (1, 2) | reward -0.04 | target = -0.04 + 0.9*0.500 = +0.410 | table[(1, 1), right]: -0.020 -> +0.195
  (1, 2) go down  -> (2, 2) | reward +1.00 | target = +1.00 + 0.9*0.000 = +1.000 | table[(1, 2), down]: +0.500 -> +0.750


**How to read that.**
- **Lap 1:** the first three steps barely change anything (the targets are about -0.04, because every number in the table is still 0). On the last step the rat reaches the cheese and gets the real **+1.0**, so the number for (square (1,2), go down) jumps up. *That is the only place real good news enters the table.*
- **Lap 2:** now the step before has a target that includes the good number next door, so it moves, and the good news creeps **one square backward per lap**.
- That backward creep is how a reward that only appears at the very end ends up guiding the very first move.

**Seeing the table as a map.** A table of 36 numbers is hard to read, but each row's best direction is just an arrow. The helper below draws the "policy" (the best direction in every square) on the maze. Squares where all four numbers are still 0 show `?`: the rat knows nothing about them.

In [6]:
def show_policy(Q, grid, path=()):
    # WHAT: for every open square, print an arrow for the direction with the biggest number in its row.
    # WHY:  THIS IS THE POLICY MADE FROM THE TABLE. "Pick the biggest number in the current square's row" is the rule that
    #       gets plugged into the `action = policy(state)` slot. Drawing it makes the table readable.
    n, m = np.array(grid).shape
    for r in range(n):
        row = ""
        for c in range(m):
            if (r, c) == (n - 1, m - 1):    row += " C"
            elif grid[r][c] == 0:           row += " █"
            else:
                values = [Q[(r, c, a)] if isinstance(Q, dict) else Q[r, c, a] for a in range(4)]
                row += " ?" if max(values) == min(values) else " " + ARROWS[int(np.argmax(values))]
        print(row)


print("after the 3 forced laps (? = never learned anything about that square):")
show_policy(Q3, GRID3)

after the 3 forced laps (? = never learned anything about that square):
 ← ↓ █
 █ → ↓
 █ █ C


**Why does the start square point left (into the wall)?** In that square the only number that changed is "go right", and it became slightly *negative* (about -0.035, because an ordinary move costs -0.04). The other three numbers are still exactly 0, and 0 is bigger than -0.035, so "pick the biggest number" picks one of the directions that was never tried. The forced laps never let the rat try anything else, so this is not a mistake in the table. It is also a preview of a quirk explained in Part 4: when every ordinary move has a small cost, **untried directions look better than tried ones**.

Now let the rat learn **on its own**, with **exploration (job 5)**: with probability `epsilon` it ignores the table and goes in a random direction, otherwise it trusts the table. Ties (like an all-zero row) are broken randomly. This is the loop you will see in any Q-learning code.

In [7]:
def best_direction(row):
    # WHAT: the policy for one square: take the direction with the biggest number; break ties at random.
    # SAME AS: CartPole notebook's `best_action_np`. BECOMES (Part 5): `np.argmax(model(state))`, where the network's 4 outputs play the role of `row`.
    best = np.flatnonzero(row == row.max())
    return int(random.choice(list(best)))


def train_table(grid, episodes, alpha, gamma, epsilon, slip=0.0):
    env = Maze(grid, slip=slip)
    Q = np.zeros((env.nrows, env.ncols, 4))            # SAME AS: the dictionary above, as an array: Q[row, col, action]
    wins = []
    for ep in range(episodes):
        state = env.reset()
        while True:
            # ---- JOB 5: EXPLORATION ----
            if random.random() < epsilon:
                action = random.randrange(4)           # WHAT: with probability epsilon, ignore the table and try something random
            else:
                action = best_direction(Q[state])      # WHAT: otherwise trust the table (JOB 2: the policy)

            reward, status = env.step(action)          # JOB 1: environment loop
            next_state = env.pos

            future = 0.0 if status != "playing" else Q[next_state].max()   # SAME AS the lap code: nothing comes after game over
            target = reward + gamma * future                                # SAME AS: target = reward + gamma * best_next
            old = Q[state + (action,)]
            Q[state + (action,)] = old + alpha * (target - old)             # SAME AS: new = old + alpha*(target - old)

            state = next_state
            if status != "playing":
                break
        wins.append(1 if env.status == "win" else 0)
    return Q, wins


random.seed(0)
Q3a, wins3 = train_table(GRID3, episodes=60, alpha=0.5, gamma=0.9, epsilon=0.2)
print("3x3 policy after 60 learning games:")
show_policy(Q3a, GRID3)
print()
print("table numbers along the route (the cell you're in, direction -> learned value):")
for cell, action in [((0, 0), RIGHT), ((0, 1), DOWN), ((1, 1), RIGHT), ((1, 2), DOWN)]:
    print(f"  {cell} {NAMES[action]:5s} -> {Q3a[cell + (action,)]:+.3f}")

3x3 policy after 60 learning games:
 → ↓ █
 █ → ↓
 █ █ C

table numbers along the route (the cell you're in, direction -> learned value):
  (0, 0) right -> +0.621
  (0, 1) down  -> +0.734
  (1, 1) right -> +0.860
  (1, 2) down  -> +1.000


**What to notice.** The arrows should form a chain from the start to the cheese: right, down, right, down. The numbers along that route **rise** as you get closer to the cheese. That rise is the same backward creep you watched by hand, repeated until it settles.

This is the same backward creep as the hallway example in the CartPole notebook, just on a grid instead of a line.

# Part 3: The same code on the 7x7 maze

> **You are here:** you saw the table learn a 3x3 maze.
> **Same as before:** `train_table` is *unchanged*. Only the grid changes: 49 squares × 4 directions = 196 numbers.
> **Why this is easy for a table:** a maze state is just `(row, col)`, and there are only 49 of them. CartPole needed bucketing because its states were decimals; here there is nothing to bucket.

In [8]:
random.seed(0)
Q7, wins7 = train_table(GRID7, episodes=600, alpha=0.1, gamma=0.95, epsilon=0.1)
# WHAT: alpha = 0.1 (small steps), gamma = 0.95 (the future counts a lot, because the cheese is ~14 moves away), epsilon = 0.1 (explore 10%)

for i in range(0, 600, 100):
    print(f"games {i+1:4d}-{i+100:4d}: rat reached the cheese in {np.mean(wins7[i:i+100])*100:5.1f}% of its games (exploring 10% of moves)")

print()
print("the learned policy (what the table says to do in every square):")
show_policy(Q7, GRID7)

games    1- 100: rat reached the cheese in  98.0% of its games (exploring 10% of moves)
games  101- 200: rat reached the cheese in 100.0% of its games (exploring 10% of moves)
games  201- 300: rat reached the cheese in 100.0% of its games (exploring 10% of moves)
games  301- 400: rat reached the cheese in 100.0% of its games (exploring 10% of moves)
games  401- 500: rat reached the cheese in 100.0% of its games (exploring 10% of moves)
games  501- 600: rat reached the cheese in 100.0% of its games (exploring 10% of moves)

the learned policy (what the table says to do in every square):
 → ↓ █ → → → ↓
 █ ↓ █ ↑ █ █ ↓
 ↓ ← ← ↑ █ ↓ ←
 ↓ █ █ █ █ ↓ █
 → → → ↓ █ ↓ ←
 █ █ █ → → ↓ █
 ? ? ? █ █ → C


In [9]:
def table_policy(state):
    return best_direction(Q7[state])                   # WHAT: the learned rule: look up the row for this square, pick the biggest number
                                                       # SAME AS: Part 1's plug-in slot, now filled with a LEARNED rule (no exploring here)

def follow_policy(Q, grid, cap=60):
    # WHAT: walk the rat using the table (no random moves) and remember the squares it visits.
    env = Maze(grid); state = env.reset(); path = [state]
    while env.status == "playing" and len(path) <= cap:
        env.step(int(np.argmax(Q[state]))); state = env.pos; path.append(state)
    return env.status == "win", path

random.seed(1)
after_7 = play_many(table_policy, GRID7)
won, path = follow_policy(Q7, GRID7)
print(f"BEFORE (random rat): wins {before_7[0]*100:5.1f}%, average {before_7[1]:5.1f} moves")
print(f"AFTER  (table rat) : wins {after_7[0]*100:5.1f}%, average {after_7[1]:5.1f} moves   (shortest possible: {shortest_path_length(GRID7)})")
print()
print(f"following the table from the start: reached the cheese = {won}, in {len(path)-1} moves. The route (*):")
show(GRID7, path=set(path))

BEFORE (random rat): wins   1.0%, average  58.0 moves
AFTER  (table rat) : wins 100.0%, average  14.0 moves   (shortest possible: 14)

following the table from the start: reached the cheese = True, in 14 moves. The route (*):
 * * █ · · · ·
 █ * █ · █ █ ·
 * * · · █ · ·
 * █ █ █ █ · █
 * * * * █ · ·
 █ █ █ * * * █
 · · · █ █ * C


**What to notice.** The three `?` squares in the bottom-left are a **sealed-off pocket**: the rat can never get in or out of it, so the table never learns anything there. Now compare "before" and "after", and compare the number of moves with the shortest possible number printed above. The maze has a **short route and a long detour**; the step cost of -0.04 makes every extra move slightly worse, so the table's numbers end up favouring the short one. Nobody told the rat which route is shorter. It found out by collecting the small penalties.

# Part 4: The knobs, tested (alpha, gamma, epsilon)

You asked earlier about the knobs. Instead of just describing them, here they are on the 7x7 maze. Each row below is a fresh rat trained for 600 games with one setting changed, repeated for 3 random seeds. "Solves it" means that following the finished table (no random moves) reaches the cheese.

In [10]:
def knob_test(label, seeds=(0, 1, 2), **settings):
    solved, moves, last_win = 0, [], []
    for s in seeds:
        random.seed(s)
        Q, w = train_table(GRID7, episodes=600, **settings)
        ok, path = follow_policy(Q, GRID7)
        solved += ok
        moves.append(len(path) - 1)
        last_win.append(np.mean(w[-100:]))
    print(f"{label:30s} solves it in {solved}/{len(seeds)} runs | moves for each run {moves} | training win rate in last 100 games {np.mean(last_win)*100:5.1f}%")

base = dict(alpha=0.1, gamma=0.95, epsilon=0.1)
knob_test("base settings", **base)
knob_test("gamma = 0.0 (ignore future)", **{**base, "gamma": 0.0})
knob_test("gamma = 0.5", **{**base, "gamma": 0.5})
knob_test("epsilon = 0.0 (never explore)", **{**base, "epsilon": 0.0})
knob_test("epsilon = 0.5 (explore a lot)", **{**base, "epsilon": 0.5})
knob_test("alpha = 1.0 (jump all the way)", **{**base, "alpha": 1.0})
knob_test("alpha = 0.01 (tiny steps)", **{**base, "alpha": 0.01})

base settings                  solves it in 3/3 runs | moves for each run [14, 14, 14] | training win rate in last 100 games 100.0%


gamma = 0.0 (ignore future)    solves it in 0/3 runs | moves for each run [60, 60, 60] | training win rate in last 100 games  76.0%


gamma = 0.5                    solves it in 3/3 runs | moves for each run [14, 14, 14] | training win rate in last 100 games 100.0%


epsilon = 0.0 (never explore)  solves it in 3/3 runs | moves for each run [14, 14, 14] | training win rate in last 100 games 100.0%


epsilon = 0.5 (explore a lot)  solves it in 3/3 runs | moves for each run [14, 14, 14] | training win rate in last 100 games 100.0%


alpha = 1.0 (jump all the way) solves it in 3/3 runs | moves for each run [14, 14, 14] | training win rate in last 100 games 100.0%


alpha = 0.01 (tiny steps)      solves it in 2/3 runs | moves for each run [14, 14, 60] | training win rate in last 100 games 100.0%


**How to read it honestly.** In this maze most settings work. That is a real finding, not a bug: a 49-square, perfectly predictable maze is an easy problem, and easy problems forgive bad knobs. Three rows deserve an explanation:

- **alpha = 0.01:** the steps are so tiny that the +1.0 at the cheese leaks backward very slowly. If a run fails, 600 games simply weren't enough for the good news to reach the start (I haven't tested giving it more games). Too small an `alpha` is slow, not wrong.

- **gamma = 0:** the rat only counts the reward from the very next move, so it can learn "don't bump into walls" but has **no idea where the cheese is**, because the +1.0 never leaks backward. `gamma` is what makes the backward creep from Part 2 possible.
- **epsilon = 0:** with no random moves you might expect the rat to get stuck, yet it may still solve the maze. The reason is a side effect: every ordinary move costs -0.04, so a direction the rat has *tried* gets a negative number, while a direction it has *never tried* is still 0, which looks better. So "pick the biggest number" keeps steering it toward things it hasn't tried yet. This is an accident of the reward design. Don't count on it in other problems.

### "You can't move partially in a maze": alpha, tested
The *rat* always moves a whole square. Only the **number in the table** moves partially. In the predictable maze above, `alpha = 1.0` was fine. Partial steps matter when **one observation can mislead you**. Below, the floor is slippery: 30% of the time a move goes in a random direction. That makes the same (square, direction) give different results on different tries, so a table entry that jumps all the way to the latest result keeps being yanked around.

In [11]:
def slippery_test(alpha, seeds=(0, 1, 2), slip=0.3):
    wins_all, moves_all = [], []
    for s in seeds:
        random.seed(s)
        Q, _ = train_table(GRID7, episodes=1500, alpha=alpha, gamma=0.95, epsilon=0.1, slip=slip)

        def greedy(state):
            return int(np.argmax(Q[state]))            # WHAT: follow the finished table, but the floor is STILL slippery when we test
        w, m = play_many(greedy, GRID7, games=200, slip=slip)
        wins_all.append(w); moves_all.append(m)
    print(f"alpha = {alpha:4}: wins {np.mean(wins_all)*100:5.1f}% of slippery games (per run: {[round(x*100) for x in wins_all]}%), average {np.mean(moves_all):5.1f} moves")

for a in (1.0, 0.5, 0.1):
    slippery_test(a)

alpha =  1.0: wins  71.8% of slippery games (per run: [100, 16, 100]%), average  30.8 moves


alpha =  0.5: wins  35.5% of slippery games (per run: [100, 1, 6]%), average 117.5 moves


alpha =  0.1: wins 100.0% of slippery games (per run: [100, 100, 100]%), average  20.2 moves


**What to notice.** Look at the per-run numbers in brackets, not just the averages. With only 3 runs per row, the averages for `alpha = 1.0` and `alpha = 0.5` are not a meaningful ordering (in an earlier test with different random seeds, 0.5 did fine and 1.0 did badly). The pattern that held in both tests is about **reliability**: the large alphas each had runs that failed badly, while `alpha = 0.1` was perfect in every run. With a large `alpha`, each table entry mostly copies the latest, possibly slipped, outcome, so the learned arrows can end up wrong; a small `alpha` averages many outcomes. That is the same "trust dial" idea as the noisy-cell demo in the CartPole notebook, now in a maze: **alpha controls how much one new observation can overwrite what you already believed.**

# Part 5: A Keras network instead of the table (DQN)

> **You are here:** a table that works on a 49-square maze.
> **New idea:** replace the table with a **neural network** that takes a *picture of the maze with the rat drawn in* (49 numbers) and outputs 4 numbers: the score for left, up, right, down. That is **one row of the table, computed instead of looked up**.
> **Same as before:** jobs 1 to 5, the target formula, epsilon exploration. **Only job 4 (the update) changes.**
> **Why bother, if the table works?** A table needs a row for every situation, which explodes for bigger mazes, and it can't guess about a situation it hasn't seen. A network generalizes: similar pictures give similar outputs.

**Two extras that real DQN code adds** (look for them in yours):
- **Replay memory:** remember each move `(picture, action, reward, next picture, game over?)` and train on a **random batch** of remembered moves, not the move just made. Consecutive moves look alike, so training on them in order is unstable; random batches fix that and reuse old experience.
- **Random starting squares:** each training game starts the rat on a random open square, so the network sees the whole maze, not just the first corridor.

**A deliberate simplification.** The popular Qmaze tutorial also paints the cells the rat has already visited into the picture and charges **-0.25 for revisiting** a cell. I left that out here, so the network faces exactly the same problem as the table (the state is just where the rat is). In a quick earlier test of the visited-cells version I saw it learn noticeably slower: after 700 games it was still winning only about two thirds of its training games, while the simpler version was already winning nearly every game it could win. I did not repeat that comparison carefully (one run each), so treat it as a hint, not a measurement.

In [12]:
import keras
import tensorflow as tf
keras.utils.set_random_seed(0)
random.seed(0)

SIZE = np.array(GRID7).size                             # 49 cells
def cells_that_can_reach_cheese(grid):
    # WHAT: flood-fill outward from the cheese to find every open square connected to it.
    # WHY:  the 7x7 maze has a sealed-off pocket (bottom-left) the rat can never leave. A game started there can never be won,
    #       so it would only add unwinnable games to the training and make the "solves it from every square" check impossible.
    g = np.array(grid); n, m = g.shape
    seen = {(n - 1, m - 1)}; stack = [(n - 1, m - 1)]
    while stack:
        r, c = stack.pop()
        for dr, dc in MOVES.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < m and g[nr, nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc)); stack.append((nr, nc))
    return seen

free_cells = sorted(cells_that_can_reach_cheese(GRID7) - {(6, 6)})   # WHAT: the squares a training game may start on
sealed_off = [(r, c) for r in range(7) for c in range(7) if GRID7[r][c] == 1 and (r, c) not in cells_that_can_reach_cheese(GRID7)]
print("open squares the rat can start from:", len(free_cells), "| sealed-off squares left out:", sealed_off)

def picture(env):
    # WHAT: the network's input: the maze as 49 numbers. Walls 0, open cells 1, and the rat's square marked 0.5.
    # WHY:  THIS REPLACES `Q[row, col]`. A table is indexed by (row, col); a network is fed this picture instead.
    # WATCH: the rat's position is encoded as "the square whose value is 0.5", so the network has to learn to find it.
    canvas = env.grid.copy()
    canvas[env.pos] = 0.5
    return canvas.reshape(1, -1).astype(np.float32)

def build_network():
    # WHAT: 49 numbers in -> 4 numbers out ([Q(left), Q(up), Q(right), Q(down)]).
    # SAME AS: one row of the table, e.g. Q7[row, col] = [left, up, right, down].
    net = keras.Sequential([
        keras.layers.Input(shape=(SIZE,)),
        keras.layers.Dense(SIZE), keras.layers.PReLU(),      # WHAT: 49 weighted sums, then a "bend" (PReLU) so it can learn curves
        keras.layers.Dense(SIZE), keras.layers.PReLU(),      # WHY: stacking layers lets it represent more than a straight-line rule
        keras.layers.Dense(4),                               # WHAT: the 4 outputs, one per direction (no bend: scores can be negative)
    ])
    # WHAT: "mse" = the average of (target - output)^2; making it small = moving outputs toward targets.
    # SAME AS: the table update `new = old + alpha*(target - old)`, which is also "move toward the target". Adam's learning_rate plays alpha's role.
    net.compile(optimizer="adam", loss="mse")
    return net

model = build_network()

@tf.function
def q_values(x):
    return model(x, training=False)                         # WHAT: ask the network for its 4 numbers (same as reading a table row)

# Before any training: the network's 4 numbers for the start square are essentially arbitrary.
start_env = Maze(GRID7); start_env.reset()
print("untrained network, 4 numbers for the start square (left, up, right, down):", np.round(q_values(picture(start_env)).numpy()[0], 3))
print("for comparison, the trained TABLE row for the start square (left, up, right, down):", np.round(Q7[0, 0], 3))

I0000 00:00:1791844432.081208     576 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791844433.740303     576 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


open squares the rat can start from: 27 | sealed-off squares left out: [(6, 0), (6, 1), (6, 2)]
untrained network, 4 numbers for the start square (left, up, right, down): [ 1.286 -0.056  0.363 -0.467]
for comparison, the trained TABLE row for the start square (left, up, right, down): [-0.625 -0.606  0.124 -0.638]


Now the training loop. Compare it line by line with `train_table`: the comments quote the table version. It trains for a fixed **time budget** so it always finishes, and stops early if the rat solves the maze from **every** starting square.

In [13]:
TIME_BUDGET = 170          # WHAT: the most seconds we will train. It stops sooner if the rat solves the maze from every square.
gamma_dqn, epsilon_dqn, batch_size = 0.95, 0.1, 32

memory = collections.deque(maxlen=8 * SIZE)                 # WHAT: the replay memory: a list that forgets its oldest items when full
dqn_wins = []
t0 = time.time()
steps = 0

def solves_from(start, cap=100):
    env = Maze(GRID7); env.reset(start); n = 0
    while env.status == "playing" and n < cap:
        env.step(int(np.argmax(q_values(picture(env)).numpy()[0]))); n += 1
    return env.status == "win", n

stopped_early = False
episode = 0
while time.time() - t0 < TIME_BUDGET:
    episode += 1
    env = Maze(GRID7)
    env.reset(random.choice(free_cells))                    # WHAT: start on a random open square
    state = picture(env)
    while True:
        # ---- JOB 2 + 5: POLICY and EXPLORATION ----
        if random.random() < epsilon_dqn:
            action = random.randrange(4)                    # SAME AS: table `random.randrange(4)` (explore)
        else:
            action = int(np.argmax(q_values(state).numpy()[0]))   # SAME AS: table `best_direction(Q[state])` (read a row, pick the biggest)

        reward, status = env.step(action)                   # JOB 1
        next_state = picture(env)

        # WHAT: remember this move. WATCH: the last item says "was the game over?" (win or lose), like `status != "playing"` above.
        memory.append((state[0], action, reward, next_state[0], float(status != "playing")))
        state = next_state
        steps += 1

        # ---- JOBS 3 + 4: TARGET and UPDATE, on a random batch from memory ----
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)       # WHAT: 32 RANDOM remembered moves (not just the latest one)
            S  = np.array([m[0] for m in batch], dtype=np.float32)
            A  = np.array([m[1] for m in batch])
            R  = np.array([m[2] for m in batch], dtype=np.float32)
            S2 = np.array([m[3] for m in batch], dtype=np.float32)
            D  = np.array([m[4] for m in batch], dtype=np.float32)

            q_now  = q_values(S).numpy()                    # WHAT: current guesses for all 32 pictures: shape (32, 4)
            q_next = q_values(S2).numpy()                   # WHAT: guesses for the NEXT pictures: shape (32, 4)

            targets = q_now.copy()                          # WHAT: start the answer key = current guesses...
            # WHAT: ...then replace ONLY the number for the direction actually taken.
            # SAME AS: table `target = reward + gamma * Q[next_state].max()` with the "game over: no future" rule written as `* (1.0 - D)`.
            # WHY:     we only have evidence about the direction we took, so the other 3 outputs keep their current value and get no push.
            targets[np.arange(batch_size), A] = R + gamma_dqn * q_next.max(axis=1) * (1.0 - D)

            # WHAT: nudge ALL the network's weights so its outputs move toward `targets`.
            # SAME AS: table `Q[state + (action,)] = old + alpha * (target - old)`.
            # DIFFERENCE: the table changed ONE number. The network changes MANY weights shared by all squares, so learning about one
            #             square also shifts its guesses for similar squares. That sharing is the point of using a network.
            model.train_on_batch(S, targets)

        if status != "playing":
            break

    dqn_wins.append(1 if env.status == "win" else 0)
    if episode % 100 == 0:
        print(f"games {episode-99:4d}-{episode:4d}: reached the cheese in {np.mean(dqn_wins[-100:])*100:5.1f}% of games   ({time.time()-t0:4.0f}s)")
    if len(dqn_wins) >= 10 and all(dqn_wins[-10:]) and all(solves_from(s)[0] for s in free_cells):
        stopped_early = True
        break

print()
print(f"trained for {episode} games, {steps} moves, {time.time()-t0:.0f}s.", "Stopped early: solves the maze from every square." if stopped_early else "Used the whole time budget.")

games    1- 100: reached the cheese in  16.0% of games   (  27s)



trained for 188 games, 10299 moves, 58s. Stopped early: solves the maze from every square.


In [14]:
def dqn_policy(state):
    # WHAT: the learned DQN rule: draw the picture for this square, ask the network, take the biggest of its 4 numbers.
    # SAME AS: Part 3's `table_policy`, with the network's 4 outputs standing in for the table's row.
    env = Maze(GRID7); env.reset(state)
    return int(np.argmax(q_values(picture(env)).numpy()[0]))

random.seed(1)
after_dqn = play_many(dqn_policy, GRID7)
ok_start, moves_start = solves_from((0, 0))
cells_solved = sum(solves_from(s)[0] for s in free_cells)

print(f"BEFORE (random rat)   : wins {before_7[0]*100:5.1f}%, average {before_7[1]:5.1f} moves")
print(f"AFTER  (table rat)    : wins {after_7[0]*100:5.1f}%, average {after_7[1]:5.1f} moves")
print(f"AFTER  (Keras rat)    : wins {after_dqn[0]*100:5.1f}%, average {after_dqn[1]:5.1f} moves   (shortest possible: {shortest_path_length(GRID7)})")
print()
print(f"Keras rat from the start square: reaches the cheese = {ok_start}, in {moves_start} moves")
print(f"Keras rat solves the maze from {cells_solved} of {len(free_cells)} possible starting squares")
print()
print("the network's 4 numbers for the start square (left, up, right, down):", np.round(q_values(picture(start_env)).numpy()[0], 3))
print("the TABLE's row for the start square                              :", np.round(Q7[0, 0], 3))

BEFORE (random rat)   : wins   1.0%, average  58.0 moves
AFTER  (table rat)    : wins 100.0%, average  14.0 moves
AFTER  (Keras rat)    : wins 100.0%, average  18.0 moves   (shortest possible: 14)

Keras rat from the start square: reaches the cheese = True, in 18 moves
Keras rat solves the maze from 27 of 27 possible starting squares

the network's 4 numbers for the start square (left, up, right, down): [-0.648 -0.513 -0.053 -0.475]
the TABLE's row for the start square                              : [-0.625 -0.606  0.124 -0.638]


**What to notice.**
- Compare the **network's 4 numbers for the start square** with the table's row at the bottom of the printout. Check which direction has the biggest number in each: they should usually agree. The sizes won't match exactly, because both are learned estimates and the network's are rougher.
- Look at "start squares solved". A network is a *function approximator*: it can be good in most squares and still wrong in a few. The table, by contrast, has a separate number for each square. (The sealed-off pocket from Part 3 is excluded from the starting squares, because nobody can win from there.)
- Compare the Keras rat's number of moves from the start with the table's and with the shortest possible. A network can find a route that works without finding the shortest one.
- Compare the training effort. The table learned the whole maze in well under a second, while the network needed about a minute (see the printed time). For a maze this small the network is overkill. Networks pay off when a table is impossible (CartPole's decimals, a maze drawn as a camera image), not here.

# Part 6: Putting it together, and finding the same jobs in other Qmaze code

## The five jobs, two ways
| Job | Q-table (Parts 2–3) | Keras network (Part 5) |
|---|---|---|
| **State** | `(row, col)` | a 49-number picture of the maze with the rat marked |
| **Where the knowledge lives** | 196 numbers in an array | the weights of the network |
| **2. Policy** | biggest number in the square's row | biggest of the network's 4 outputs |
| **5. Exploration** | `epsilon` | `epsilon` (same) |
| **3. Target** | `reward + gamma * max Q[next square]` (0 if game over) | the same formula, written `R + gamma * q_next.max(axis=1) * (1 - D)` |
| **4. Update** | change **one number** a fraction `alpha` toward the target | nudge **all weights** toward the target (`train_on_batch`) |
| **Learns from** | each move, immediately | random batches from a **replay memory** |

## The same jobs in the widely shared Qmaze tutorial
Its names are usually these. In code that names things differently, search for the *job* in the right-hand column.

| Look for | It is job… |
|---|---|
| a `Qmaze` class with `act(action)`, `get_reward()`, `game_status()`, `observe()` | 1: the environment (my `Maze`, `step`, the reward lines, `picture`) |
| `valid_actions()` | a helper listing moves that don't hit a wall |
| an `Experience` class with `remember(...)` and `get_data(...)` | the replay memory and the target (my `memory`, `batch`, `targets` lines) |
| `if np.random.rand() < epsilon:` | 5: exploration |
| `np.argmax(...)` on the model's prediction | 2: policy |
| `reward + discount * np.max(...)`, with a game-over case | 3: target |
| `model.fit(...)` | 4: update |
| a loop that stops when the win rate is high and every starting cell is solved | when to stop (my "solves from every square" check) |

## Honest limits of this notebook
- Each learner was trained **once**, so don't treat the Keras-versus-table comparison as a fair race. It shows that both learn, and how they differ in *how*.
- This is a **predictable** maze (except in the slippery test). Real tasks are noisier, which is when alpha, gamma and epsilon start to matter much more than they did here.